In [ ]:
import os, sys, math
# Runtime only: put the v6 repo dir on sys.path so `import modules` works from
# notebooks/. Editor go-to-definition comes from python.analysis.extraPaths.
_V6 = os.path.abspath(".." if os.path.isdir("../modules") else ".")
if _V6 not in sys.path:
    sys.path.insert(0, _V6)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import modules  # noqa: F401 — sys.path injection

# A kernel keeps every module it has already imported in sys.modules, so editing a
# pipeline module on disk mid-session does NOT take effect on re-run. Reload the
# pipeline modules (dependencies first) so re-running this cell always tracks the
# current source -- see explore_trigger_counts.ipynb for the ImportError this
# guards against when a module has moved.
import importlib
for _m in ("modules.constants", "modules.geometry", "modules.layouts.strategies",
           "modules.surrogates", "modules.optimize"):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from modules.geometry import SurfaceUpMap, load_tr_mountain
from modules.layouts.strategies import _STRATEGIES, _STRATEGY_FNS
from modules.surrogates.fnn import encode_primary
from modules.surrogates.dual import load_dual_surrogate
from modules.optimize import LAYOUT_THRESHOLD
from modules.constants import (
    GEOMETRY_PATH_RESOLVED, GEOMETRY_GROUP, DET_KEY,
    EAST_ENTRY, LAYER_EAST_DX, N_PLANES, N_DETECTORS, SIGMA_SPATIAL,
    DUAL_SHOWER_CACHE_PATH, DUAL_POSITIONS_PATH, DUAL_SPECIES_IDS_PATH,
    LOG_E_MIN, LOG_E_MAX, T_LOG_SCALE, FNN_FOLDER,
)
import showerdata

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device={DEVICE}  LAYOUT_THRESHOLD={LAYOUT_THRESHOLD}  SIGMA_SPATIAL={SIGMA_SPATIAL}')

mountain = load_tr_mountain(GEOMETRY_PATH_RESOLVED, GEOMETRY_GROUP, DET_KEY,
                            east_entry=EAST_ENTRY, layer_east_dx=LAYER_EAST_DX,
                            n_planes=N_PLANES)
surface  = SurfaceUpMap.from_mountain(mountain, grid_h=256, grid_w=256).to(DEVICE)
_mE, _mN, _mU = (mountain.centroids_ENU[:, i] for i in range(3))
print(f'mountain  East=[{_mE.min():.0f}, {_mE.max():.0f}]  North=[{_mN.min():.0f}, {_mN.max():.0f}]'
      f'  Up=[{_mU.min():.0f}, {_mU.max():.0f}]')

# ── Stage 2: the CURRENT trained surrogate checkpoint ────────────────────────
# load_dual_surrogate reads fnn_<species>.pt for every entry in SPECIES_NAMES out
# of FNN_FOLDER (modules.constants.RUN_LOCATION's run) and prints each
# checkpoint's own path + mtime as it loads -- that printed provenance line is
# what "current checkpoint" means in this notebook's title.
dual = load_dual_surrogate(FNN_FOLDER, DEVICE)
print(f"[dual] supports_var={dual.supports_var}  "
      f"(forward_sample below needs this True; falls back to the mean otherwise)")


## Pick a diverse set of showers

Stage 2 (`02_train_fnn_deepsets.py`) predicts detector response from a **primary
encoding + a layout** — `dual.forward(primary, xy) -> (E, T)` — it never sees the raw
shower point cloud. So unlike `explore_trigger_counts.ipynb`, this notebook never streams
`showerdata.load(...)`'s heavy point-cloud tensors: `showerdata.load_inc_particles` reads
only the per-shower metadata (direction, energy, EM/hadronic class), the same call
`modules.data.dataset_builder._load_corpus_metadata` makes to build `primary.pt` in Step 1.


In [ ]:
N_SHOWERS_NB = 300            # showers pulled into the notebook

# Metadata only (direction, energy, EM/hadronic class) -- no point clouds.
meta = showerdata.load_inc_particles(DUAL_SHOWER_CACHE_PATH)
species_all = torch.load(DUAL_SPECIES_IDS_PATH)               # (M,) idx into SPECIES_NAMES
n_pairs = int((species_all == 0).sum())                       # electron-block size
sl = slice(0, min(N_SHOWERS_NB, n_pairs))

# The primary (direction/energy/EM-hadronic-class/decay-vertex) is shared by all
# species components of one physical event (§3.6 in docs/THEORY.md), so slicing
# just the electron block gives the same primaries every species would.
dirs   = torch.as_tensor(meta.directions[sl.start:sl.stop], dtype=torch.float32)
energs = torch.as_tensor(meta.energies[sl.start:sl.stop],   dtype=torch.float32).reshape(-1)
pdg    = torch.as_tensor(meta.pdg[sl.start:sl.stop],        dtype=torch.long)
positions_all = torch.load(DUAL_POSITIONS_PATH)                # (M,3) ENU East,North,Up
positions = positions_all[sl.start:sl.stop].float()             # (S,3) real decay vertices
dirs_u = dirs / dirs.norm(dim=1, keepdim=True).clamp(min=1e-12) # unit travel dirs

# Array centre from the mesh (modules.data.dataset_builder's own convention),
# not a constant, so encode_primary's rel_E/N/U tracks the current geometry.
array_center = torch.as_tensor(mountain.centroids_ENU, dtype=torch.float32).mean(dim=0)
primaries = encode_primary(dirs_u, energs, pdg, positions, array_center).to(DEVICE)  # (S, PRIMARY_DIM)
print(f'corpus: {n_pairs} pairs; encoded {tuple(primaries.shape)} primaries '
      f'(E {float(energs.min()):.2e}..{float(energs.max()):.2e} GeV)')


## Screen on a default grid layout, using the surrogate's predicted response

Same purpose as `explore_trigger_counts.ipynb`'s screen — pick primaries spanning energy
and zenith, from those the array actually sees something from — except "sees something"
here means the **surrogate** predicts a nonzero deposit, not the kernel. This screen uses
`dual.forward(...)`, the deterministic mean, since it is only choosing a diverse set; the
representative-pattern figures below switch to `dual.forward_sample(...)`, the stochastic
draw Stage 3 trains recon on and Stage 4 optimizes against by default (`docs/THEORY.md`
§10.4).


In [ ]:
rng_scan = np.random.default_rng(42)
fn_grid  = _STRATEGY_FNS['layout_grid']
e_scan, n_scan = fn_grid(mountain, n_det=N_DETECTORS, rng=rng_scan)   # (East, North)
e_scan = e_scan.float().to(DEVICE); n_scan = n_scan.float().to(DEVICE)
xy_scan = torch.stack([e_scan, n_scan], dim=-1).unsqueeze(0)          # (1, n_det, 2)

scan_records = []
with torch.no_grad():
    for s in range(primaries.shape[0]):                      # every loaded shower
        pred = dual.forward(primaries[s:s+1], xy_scan)        # (1, n_det, 2), mean, log-encoded
        E0 = torch.expm1(pred[0, :, 0]).clamp(min=0)          # raw counts (col 0 = log1p(N_tot))
        # theta/phi in the pipeline's own convention -- see explore_trigger_counts.ipynb's
        # matching cell for why this reads directly off dirs_u.
        theta = float(torch.arccos(dirs_u[s, 2].clamp(-1.0, 1.0)).item())
        phi   = float(torch.atan2(dirs_u[s, 1], dirs_u[s, 0]).item())
        scan_records.append(dict(
            shower=s, E=float(energs[s].item()), theta_deg=math.degrees(theta),
            phi_deg=math.degrees(phi) % 360.0,
            n_trig=int((E0 > LAYOUT_THRESHOLD).sum().item()),   # above the 04 cut
            n_nonzero=int((E0 > 0).sum().item()),               # any predicted response
            e_sum=float(E0.sum().item()),
            e_max=float(E0.max().item()),
        ))
df_scan = pd.DataFrame(scan_records)
n = len(df_scan)
print(f"scanned {n} showers on the default grid (surrogate mean prediction)")
print(f"  any detector > LAYOUT_THRESHOLD={LAYOUT_THRESHOLD}: "
      f"{int((df_scan.n_trig > 0).sum())}/{n} ({100*(df_scan.n_trig > 0).mean():.1f}%)")
print(f"  any nonzero predicted detector response  : "
      f"{int((df_scan.n_nonzero > 0).sum())}/{n} ({100*(df_scan.n_nonzero > 0).mean():.1f}%)")
print(f"  max single-detector predicted response over all: {df_scan.e_max.max():.4g}")
df_scan[['E', 'theta_deg', 'phi_deg', 'n_trig', 'n_nonzero', 'e_sum']].describe()


## Pick 6 showers spanning E and θ, from those the surrogate predicts a nonzero response for


In [ ]:
# Screening on `n_trig > 0` would leave a thin (often empty) sample -- select on
# e_sum > 0 (the surrogate predicts the array sees something) and report n_trig
# alongside, same rule explore_trigger_counts.ipynb uses for the kernel.
df_hit = df_scan[df_scan['e_sum'] > 0].copy()
print(f"{len(df_hit)}/{len(df_scan)} showers get a nonzero predicted response; "
      f"{int((df_hit.n_trig > 0).sum())} of those clear LAYOUT_THRESHOLD")
if df_hit.empty:
    raise RuntimeError("no shower got a nonzero prediction — raise N_SHOWERS_NB in the setup cell")

df_hit['logE'] = np.log10(df_hit['E'])
_qE  = min(3, df_hit['logE'].nunique())
_qth = min(2, df_hit['theta_deg'].nunique())
df_hit['E_bin']     = pd.qcut(df_hit['logE'], q=_qE,
                              labels=['Elow', 'Emid', 'Ehigh'][:_qE], duplicates='drop')
df_hit['theta_bin'] = pd.qcut(df_hit['theta_deg'], q=_qth,
                              labels=['vert', 'horiz'][:_qth], duplicates='drop')

selected = []
for E_b in df_hit['E_bin'].cat.categories:
    for th_b in df_hit['theta_bin'].cat.categories:
        sub = df_hit[(df_hit['E_bin'] == E_b) & (df_hit['theta_bin'] == th_b)]
        if not len(sub):
            continue
        # Prefer the strongest depositor in the cell so the pattern plots show signal.
        pick = sub.sort_values('e_sum', ascending=False).iloc[0]
        selected.append(dict(shower=int(pick['shower']), E=float(pick['E']),
                             theta_deg=float(pick['theta_deg']),
                             phi_deg=float(pick['phi_deg']),
                             n_trig_default=int(pick['n_trig']),
                             e_sum=float(pick['e_sum']),
                             tag=f'{E_b}/{th_b}'))
df_show = pd.DataFrame(selected)
df_show


## For each shower × strategy, reinitialize N_SEEDS times — surrogate's stochastic prediction

Same reinitialization sweep as `explore_trigger_counts.ipynb`, but each (shower, strategy,
seed) draws its detector response from `dual.forward_sample(...)` — the reparameterized
sample from the surrogate's predicted (mean, var), the same draw Stage 3 trains recon on
and Stage 4 optimizes against by default (`sample=True`, `docs/THEORY.md` §10.4) — rather
than the plane-aware kernel. Both `E` and `T` are decoded back to raw units (`expm1` for
counts, `expm1(...)/T_LOG_SCALE` for seconds) before being cached, so the two figures below
draw the same raw quantities the ground-truth notebook does, just from the surrogate.


In [ ]:
N_SEEDS = 15  # reinitializations per (shower, strategy)
records = []
layouts_cache = {}       # (shower, strategy, seed) -> (E (n_det,), e_det, n_det_xy)  [East, North]
layouts_cache_time = {}  # same key -> (E, T, e_det, n_det_xy)

_predict = dual.forward_sample if dual.supports_var else dual.forward
if not dual.supports_var:
    print("[warn] this checkpoint has no learned variance head -- falling back to "
          "the deterministic mean (dual.forward) instead of dual.forward_sample")

with torch.no_grad():
    for _, row in df_show.iterrows():
        s = int(row['shower'])
        q = primaries[s:s+1]
        for s_idx, (s_name, fn_name, kwargs) in enumerate(_STRATEGIES):
            fn = _STRATEGY_FNS[fn_name]
            for seed in range(N_SEEDS):
                rng = np.random.default_rng(1000 * s_idx + seed)
                e_det, n_det_xy = fn(mountain, n_det=N_DETECTORS, rng=rng, **kwargs)  # (East, North)
                e_det = e_det.float().to(DEVICE); n_det_xy = n_det_xy.float().to(DEVICE)
                xy_b = torch.stack([e_det, n_det_xy], dim=-1).unsqueeze(0)             # (1, n_det, 2)

                pred = _predict(q, xy_b)                                              # (1, n_det, 2), log-encoded
                E_arr = torch.expm1(pred[0, :, 0]).clamp(min=0)                       # raw counts
                T_arr = torch.expm1(pred[0, :, 1]).clamp(min=0) / T_LOG_SCALE         # raw seconds

                n_trig = int((E_arr > LAYOUT_THRESHOLD).sum().item())
                e_sum  = float(E_arr.sum().item())
                records.append(dict(shower=s, tag=row['tag'], strategy=s_name,
                                    seed=seed, n_trig=n_trig, e_sum=e_sum,
                                    E_prim=row['E'], theta_deg=row['theta_deg'],
                                    phi_deg=row['phi_deg']))
                layouts_cache[(s, s_name, seed)] = (E_arr.cpu().numpy(),
                                                    e_det.cpu().numpy(),
                                                    n_det_xy.cpu().numpy())
                layouts_cache_time[(s, s_name, seed)] = (E_arr.cpu().numpy(),
                                                         T_arr.cpu().numpy(),
                                                         e_det.cpu().numpy(),
                                                         n_det_xy.cpu().numpy())
df = pd.DataFrame(records)
print(f'samples: {len(df)}  ({len(df_show)} showers x {len(_STRATEGIES)} strategies x {N_SEEDS} seeds)')
df.head()


## Detector patterns, coloured by predicted energy (Stage 2 surrogate)

Same 3D ENU view and picking rule as `explore_trigger_counts.ipynb`'s ground-truth figure
(mid-`n_trig` seed of `EXAMPLE_STRATEGY`, per representative shower) — mountain in
`#DDD6C9`, the decay vertex as a red star, detectors as triangles coloured by the
**surrogate's predicted counts** (`plasma`, log scale) rather than the kernel's. There is
no shower-cloud overlay: Stage 2 never sees the point cloud, only the primary encoding, so
there is nothing of that kind to draw.


In [ ]:
import importlib.util as _ilu
from matplotlib.ticker import MaxNLocator
from plots.lib import geometry_plots

_spec = _ilu.spec_from_file_location("opt_plotting",
                                     os.path.join(_V6, "plots", "lib", "opt_plotting.py"))
_optplt = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_optplt)

MTN_ENU  = _optplt.mountain_enu(mountain)   # (n, 3) East, North, Up
SURF_CPU = surface.to("cpu")                # projection is a read-only plotting step
P_dec    = positions.numpy()                # (S, 3) real decay vertices

# Half-width [m] of the East/North box drawn around the array centre -- same
# framing choice as explore_trigger_counts.ipynb's detector-pattern figure.
VIEW_R = 1500.0
CTR = MTN_ENU.mean(axis=0)
UP_LO, UP_HI = MTN_ENU[:, 2].min() - 300.0, MTN_ENU[:, 2].max() + 300.0

def _in_view(P):
    return ((np.abs(P[:, 0] - CTR[0]) <= VIEW_R)
            & (np.abs(P[:, 1] - CTR[1]) <= VIEW_R)
            & (P[:, 2] >= UP_LO) & (P[:, 2] <= UP_HI))


class _NoClouds:
    """Passed as `points` to `plot_detector_patterns`/`_by_time`: Stage 2 never
    sees a point cloud, so every index returns None and `panel`'s `cloud=None`
    branch simply skips the cloud scatter (same branch explore_trigger_counts.py's
    panel takes when it has nothing to draw there)."""
    def __getitem__(self, i):
        return None

NO_CLOUDS = _NoClouds()


def panel(ax, E_arr, e_det, n_det_xy, cloud=None, s=None):
    """One 3D ENU panel: mountain + decay vertex + detectors by predicted count.

    `cloud` is accepted only for signature parity with
    explore_trigger_counts.ipynb's `panel` (`plot_detector_patterns` always
    passes one) -- it is always None here.
    """
    ax.scatter(MTN_ENU[:, 0], MTN_ENU[:, 1], MTN_ENU[:, 2], s=3, c="#DDD6C9",
               alpha=0.6, depthshade=False, label="mountain")
    if s is not None and _in_view(P_dec[s][None, :])[0]:
        ax.scatter(*P_dec[s], c="red", s=110, marker="*", depthshade=False,
                   label="decay vertex")

    sc, n_live, n_over = _optplt.draw_detectors_enu_3d(
        ax, E_arr, e_det, n_det_xy, SURF_CPU,
        layout_threshold=LAYOUT_THRESHOLD, log1p_space=False)

    ax.set_xlim(CTR[0] - VIEW_R, CTR[0] + VIEW_R)
    ax.set_ylim(CTR[1] - VIEW_R, CTR[1] + VIEW_R)
    ax.set_zlim(UP_LO, UP_HI)
    ax.set_xlabel("East [m]", labelpad=8)
    ax.set_ylabel("North [m]", labelpad=8)
    ax.set_zlabel("Up [m]", labelpad=2)
    ax.tick_params(axis="x", pad=2); ax.tick_params(axis="y", pad=2)
    ax.tick_params(axis="z", pad=0)
    for a in (ax.xaxis, ax.yaxis, ax.zaxis):
        a.set_major_locator(MaxNLocator(nbins=4))
    ax.view_init(elev=25, azim=-125)
    return sc, n_live, n_over

EXAMPLE_STRATEGY = 'center_gauss400'
fig = geometry_plots.plot_detector_patterns(
    panel, df, df_show, layouts_cache, points=NO_CLOUDS,
    example_strategy=EXAMPLE_STRATEGY, layout_threshold=LAYOUT_THRESHOLD)
with plt.rc_context({'savefig.bbox': 'tight', 'savefig.pad_inches': 0.3}):
    plt.show()


## Detector patterns, coloured by predicted arrival time (Stage 2 surrogate)

Same six representative (shower, layout) picks and the same 3D ENU view as the figure
above, but detectors are coloured by the **surrogate's predicted arrival time** instead of
predicted counts, on a linear (not log) scale in µs — see
`explore_trigger_counts.ipynb`'s matching time-colored cell for why linear. The dark/lit
split is still `E`-derived, so both figures here show the identical lit set; only the
colour channel differs.


In [ ]:
def panel_time(ax, E_arr, T_arr, e_det, n_det_xy, cloud=None, s=None):
    """Twin of `panel` above, coloured by the surrogate's predicted arrival time
    instead of predicted counts. `cloud` is accepted for signature parity only
    -- see `panel`."""
    ax.scatter(MTN_ENU[:, 0], MTN_ENU[:, 1], MTN_ENU[:, 2], s=3, c="#DDD6C9",
               alpha=0.6, depthshade=False, label="mountain")
    if s is not None and _in_view(P_dec[s][None, :])[0]:
        ax.scatter(*P_dec[s], c="red", s=110, marker="*", depthshade=False,
                   label="decay vertex")

    sc, n_live, n_over = _optplt.draw_detectors_enu_3d_by_time(
        ax, E_arr, T_arr, e_det, n_det_xy, SURF_CPU,
        layout_threshold=LAYOUT_THRESHOLD)

    ax.set_xlim(CTR[0] - VIEW_R, CTR[0] + VIEW_R)
    ax.set_ylim(CTR[1] - VIEW_R, CTR[1] + VIEW_R)
    ax.set_zlim(UP_LO, UP_HI)
    ax.set_xlabel("East [m]", labelpad=8)
    ax.set_ylabel("North [m]", labelpad=8)
    ax.set_zlabel("Up [m]", labelpad=2)
    ax.tick_params(axis="x", pad=2); ax.tick_params(axis="y", pad=2)
    ax.tick_params(axis="z", pad=0)
    for a in (ax.xaxis, ax.yaxis, ax.zaxis):
        a.set_major_locator(MaxNLocator(nbins=4))
    ax.view_init(elev=25, azim=-125)
    return sc, n_live, n_over

fig = geometry_plots.plot_detector_patterns_by_time(
    panel_time, df, df_show, layouts_cache_time, points=NO_CLOUDS,
    example_strategy=EXAMPLE_STRATEGY, layout_threshold=LAYOUT_THRESHOLD)
with plt.rc_context({'savefig.bbox': 'tight', 'savefig.pad_inches': 0.3}):
    plt.show()
